# Async browser tools and graph execution: practice

Create browser/toolkit → inspect tools → navigate/extract → combine tools → compile checkpointed graph → await ainvoke.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [4_langgraph/3_lab3.ipynb](../../../../../4_langgraph/3_lab3.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 2. Browser tools have setup, action, and cleanup

A browser toolkit makes actions such as navigation and extraction callable. The practice object imitates that lifetime with fixed HTML; it launches no browser and accesses no website.

**Predict:** Can extraction happen before navigation in this demonstration?

In [ ]:
class PracticeBrowser:
    def __init__(self):
        self.page = None
        self.closed = False
    def navigate(self, html):
        self.page = html
    def extract(self):
        if self.page is None:
            raise RuntimeError("Navigate first")
        return self.page.replace("<p>", "").replace("</p>", "")
    def close(self):
        self.closed = True
browser = PracticeBrowser()
try:
    browser.navigate("<p>Fictional page text</p>")
    print(browser.extract())
finally:
    browser.close()
assert browser.closed

**Check:** No in this fixture. A fake browser teaches sequence only; it does not validate Playwright or real page behavior.

**Change one thing:** Call extract before navigate and inspect the error. Explain why real Playwright also needs binaries and cleanup.

## 3. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 4. Build a real local LangGraph without a model

A graph is state plus nodes plus edges. Nodes return state updates. Compilation builds the executable workflow; invoke runs it. This exercises the installed LangGraph package locally.

**Predict:** Which value will the uppercase node return?

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    text: str
def uppercase(state):
    return {"text": state["text"].upper()}
builder = StateGraph(State)
builder.add_node("uppercase", uppercase)
builder.add_edge(START, "uppercase")
builder.add_edge("uppercase", END)
graph = builder.compile()
result = graph.invoke({"text": "hello"})
print(result)
assert result["text"] == "HELLO"

**Check:** HELLO. Graphs can orchestrate ordinary Python functions; a language model is optional.

**Change one thing:** Add a second node that appends an exclamation mark. Wire it between uppercase and END.

## 5. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.